# Final Loan Classification Analysis  
  
Checkpointed end-to-end classification workflow.  
v1.5 - updated 11.09.26  


### SETUP

In [ ]:
# Set the `RESUME_FROM` = None to reload existing checkpoints
#  OR set `RESUME_FROM` = "fresh" to run all cells from scratch

RESUME_FROM = None
# RESUME_FROM = None     # load latest checkpoint when available

from pathlib import Path
from src.config import EXPORTS_PATH
def checkpoint_exists(name):
    return any(
        Path(EXPORTS_PATH).glob(f"loan_{name}_*.pkl")
    )

def use_checkpoint(name):
    return (
        RESUME_FROM is None
        and checkpoint_exists(name)
    )

### IMPORT

In [ ]:

import time
from pathlib import Path
import pandas as pd

from src.config import (
    EXPORTS_PATH, 
    MODELS_PATH, 
    REPORTS_PATH,
    FILE_PATH,
)
from src.data_loader import (
    load_data, 
    save_progress, 
    # latest_progress,
    load_progress,
    # use_checkpoint,
)

from src.loans_preprocessing import prepare_loan_data
from src.model_pipeline import (
    split_classification_and_checkpoint,
    fit_and_checkpoint_encoders,
    train_logistic_regression,
    train_decision_tree_classifier,
    train_gradient_boosting_classifier,
    evaluate_classifier,
    update_classification_leaderboard_csv,
    log_classification_model_to_mlflow,
    load_encoder_checkpoint,

)
from src.utils import create_output_folders

create_output_folders(EXPORTS_PATH, MODELS_PATH, REPORTS_PATH)

## Step 1 - Load and Prepare Data

In [3]:
if use_checkpoint("prepared_data"):
    df_model = load_progress("prepared_data")
else:
    df_raw = load_data(Path(FILE_PATH))
    df_model = prepare_loan_data(df_raw)
    save_progress("prepared_data", df_model)
    print(f'Raw shape: {df_raw.shape}')

# if RESUME_FROM == "fresh":
#     df_raw = load_data(Path(config.FILE_PATH))
#     df_model = prepare_loan_data(df_raw)
#     save_progress("prepared_data", df_model)
# else:
#     df_model = load_progress("prepared_data")

# df_raw = load_data(Path(FILE_PATH))
# df_model = prepare_loan_data(df_raw)
# save_progress('prepared_data', df_model)

print(f'Model-ready shape: {df_model.shape}')
df_model.head()

Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_prepared_data_20260823_192937.pkl
Model-ready shape: (450, 14)


,gender,married,education,self_employed,applicant_income,coapplicant_income,loan_amount,loan_amount_term,credit_history,property_area,loan_status,has_dependents,total_income,debt_income_ratio
0,Male,Yes,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural,0.0,1,6091.0,47.585938
1,Male,Yes,Graduate,No,3000.0,0.0,66.0,360.0,1.0,Urban,1.0,0,3000.0,45.454545
2,Male,Yes,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban,1.0,0,4941.0,41.175000
3,Male,No,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban,1.0,0,6000.0,42.553191
4,Male,Yes,Graduate,Yes,5417.0,4196.0,267.0,360.0,1.0,Urban,1.0,1,9613.0,36.003745


## Step 2 - Stratified Train/Test Split

In [4]:
from src.data_config import BASELINE_MODEL_SPECS

if use_checkpoint("split_manifest"):
    split = load_progress("split_manifest")
else:
    split = split_classification_and_checkpoint(
        df_model,
        specs=BASELINE_MODEL_SPECS,
    )
    save_progress("split_manifest", split)

print(f"X_train: {split['X_train'].shape}")
print(f"X_test:  {split['X_test'].shape}")
print(split['y_train'].value_counts(normalize=True).round(3))

Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_split_manifest_20260823_194512.pkl
X_train: (360, 13)
X_test:  (90, 13)
loan_status
1.0    0.725
0.0    0.275
Name: proportion, dtype: float64


## Step 3 - Fit Encoders and Encode Features

In [5]:

if use_checkpoint("encoder_manifest"):
    encoder_manifest = load_progress("encoder_manifest")
    encoders = {
        "encoder_scaled": load_encoder_checkpoint(
            encoder_manifest["scaled_path"]
        ),
        "encoder_raw": load_encoder_checkpoint(
            encoder_manifest["raw_path"]
        ),
        **encoder_manifest,
    }
else:
    encoders = fit_and_checkpoint_encoders(
        split["X_train"],
        task="classification",
        specs=BASELINE_MODEL_SPECS,
    )
    save_progress("encoder_manifest", encoders)
    

if use_checkpoint("encoded_features"):
    encoded_features = load_progress("encoded_features")
    X_train_scaled = encoded_features["X_train_scaled"]
    X_test_scaled = encoded_features["X_test_scaled"]
    X_train_raw = encoded_features["X_train_raw"]
    X_test_raw = encoded_features["X_test_raw"]
else:
    X_train_scaled = encoders["encoder_scaled"].transform(
        split["X_train"]
    )
    X_test_scaled = encoders["encoder_scaled"].transform(
        split["X_test"]
    )
    X_train_raw = encoders["encoder_raw"].transform(
        split["X_train"]
    )
    X_test_raw = encoders["encoder_raw"].transform(
        split["X_test"]
    )

    save_progress(
        "encoded_features",
        {
            "X_train_scaled": X_train_scaled,
            "X_test_scaled": X_test_scaled,
            "X_train_raw": X_train_raw,
            "X_test_raw": X_test_raw,
        },
    )

print(f'Scaled train shape: {X_train_scaled.shape}')
print(f'Raw train shape: {X_train_raw.shape}')

Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_encoder_manifest_20260823_193003.pkl
Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_encoded_features_20260823_193003.pkl
Scaled train shape: (360, 19)
Raw train shape: (360, 19)


In [14]:
from src.model_pipeline import train_dummy_classifier
from src.config import RANDOM_SEED

start_time = time.time()

model_dummy = train_dummy_classifier(
    X_train_raw,
    split["y_train"],
    random_state=RANDOM_SEED,
)

metrics_dummy = evaluate_classifier(
    model_dummy,
    X_test_raw,
    split["y_test"],
    model_name="Dummy Majority Classifier",
    train_time_sec=time.time() - start_time,
)

update_classification_leaderboard_csv(metrics_dummy)
metrics_dummy

save_progress(
    "dummy_classifier_model",
    {
        "model": model_dummy,
        "metrics": metrics_dummy,
    },
)

Progress saved: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_dummy_classifier_model_20260823_202500.pkl


'Q:\\scripts\\projects\\Mock_Interview-July2026\\exports\\loan_dummy_classifier_model_20260823_202500.pkl'

## Step 4 - Train and Evaluate Models

In [16]:
from src.config import RANDOM_SEED, REPORTS_DIR
def train_and_save(
    checkpoint_name,
    model_name,
    trainer,
    X_train,
    X_test,
    encoder_path,
):
    if use_checkpoint(checkpoint_name):
        state = load_progress(checkpoint_name)
        return (
            state["model"],
            state["metrics"],
        )

    start_time = time.time()
    model = trainer(
        X_train,
        split["y_train"],
        random_state=RANDOM_SEED,
    )

    metrics = evaluate_classifier(
        model,
        X_test,
        split["y_test"],
        model_name=model_name,
        train_time_sec=time.time() - start_time,
    )

    save_progress(
        checkpoint_name,
        {
            "model": model,
            "metrics": metrics,
            "encoder_path": encoder_path,
        },
    )

    update_classification_leaderboard_csv(metrics)

    return model, metrics

model_logistic, metrics_logistic = train_and_save(
    "logistic_regression_model",
    "Logistic Regression",
    train_logistic_regression,
    X_train_scaled,
    X_test_scaled,
    encoders["scaled_path"],
)

model_tree, metrics_tree = train_and_save(
    "decision_tree_classifier_model",
    "Decision Tree Classifier",
    train_decision_tree_classifier,
    X_train_raw,
    X_test_raw,
    encoders["raw_path"],
)

model_gb, metrics_gb = train_and_save(
    "gradient_boosting_classifier_model",
    "Gradient Boosting Classifier",
    train_gradient_boosting_classifier,
    X_train_raw,
    X_test_raw,
    encoders["raw_path"],
)

leaderboard_path = Path(REPORTS_DIR) / 'classification_model_leaderboard.csv'
leaderboard = pd.read_csv(leaderboard_path).sort_values(['roc_auc', 'f1'], ascending=False, na_position='last').reset_index(drop=True)
leaderboard

Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_logistic_regression_model_20260823_193103.pkl
Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_decision_tree_classifier_model_20260823_193103.pkl
Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_gradient_boosting_classifier_model_20260823_193104.pkl


,model_name,accuracy,precision,recall,f1,roc_auc,train_time_sec
0,Logistic Regression,0.800000,0.805195,0.953846,0.873239,0.715077,0.022218
1,Gradient Boosting Classifier,0.711111,0.753247,0.892308,0.816901,0.701538,0.293986
2,Decision Tree Classifier,0.622222,0.738462,0.738462,0.738462,0.529231,0.007307
3,Dummy Majority Classifier,0.722222,0.722222,1.000000,0.838710,0.500000,0.001106


## Step 5 - Log Models to MLflow

In [17]:
if use_checkpoint("mlflow_runs"):
    mlflow_runs = load_progress("mlflow_runs")
else:
    mlflow_runs = {}

    for model, metrics, encoder_path in [
        (
            model_logistic,
            metrics_logistic,
            encoders["scaled_path"],
        ),
        (
            model_tree,
            metrics_tree,
            encoders["raw_path"],
        ),
        (
            model_gb,
            metrics_gb,
            encoders["raw_path"],
        ),
    ]:
        run_id = log_classification_model_to_mlflow(
            model,
            metrics,
            encoder_path,
            split["checkpoint_path"],
        )
        mlflow_runs[metrics["model_name"]] = run_id

    save_progress("mlflow_runs", mlflow_runs)

Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_mlflow_runs_20260823_193233.pkl


## Step 6 - Save Best Model Selection

In [18]:
best_model_name = leaderboard.iloc[0]['model_name']
best_model = {'Logistic Regression': model_logistic, 'Decision Tree Classifier': model_tree, 'Gradient Boosting Classifier': model_gb}[best_model_name]
best_model_state = {'name': best_model_name, 'model': best_model, 'leaderboard': leaderboard, 'mlflow_run_id': mlflow_runs.get(best_model_name)}
save_progress('best_model', best_model_state)
print(f'Current best model: {best_model_name}')
leaderboard

Progress saved: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_best_model_20260823_202539.pkl
Current best model: Logistic Regression


,model_name,accuracy,precision,recall,f1,roc_auc,train_time_sec
0,Logistic Regression,0.800000,0.805195,0.953846,0.873239,0.715077,0.022218
1,Gradient Boosting Classifier,0.711111,0.753247,0.892308,0.816901,0.701538,0.293986
2,Decision Tree Classifier,0.622222,0.738462,0.738462,0.738462,0.529231,0.007307
3,Dummy Majority Classifier,0.722222,0.722222,1.000000,0.838710,0.500000,0.001106


### Initial Exploratory Evaluation

In [21]:
from src.data_config import FEATURE_COLUMNS

raw_features = FEATURE_COLUMNS


def source_feature(encoded_feature):
    encoded_feature = encoded_feature.split("__", 1)[-1]

    matches = [
        feature
        for feature in raw_features
        if (
            encoded_feature == feature
            or encoded_feature.startswith(f"{feature}_")
        )
    ]

    if not matches:
        return encoded_feature

    # Longest match prevents loan_amount_term
    # from being incorrectly grouped as loan_amount.
    return max(matches, key=len)


feature_importance["source_feature"] = (
    feature_importance["feature"].apply(source_feature)
)

source_importance = (
    feature_importance
    .groupby("source_feature", as_index=False)
    .agg(
        importance_mean=("importance_mean", "sum"),
        importance_std=("importance_std", "mean"),
    )
    .sort_values("importance_mean", ascending=False)
)

source_importance

,source_feature,importance_mean,importance_std
2,credit_history,0.184144,0.043670
1,coapplicant_income,0.013436,0.006953
9,married,0.010708,0.014562
11,self_employed,0.002421,0.003369
12,total_income,0.000103,0.000229
0,applicant_income,-0.000944,0.001824
3,debt_income_ratio,-0.002256,0.002448
6,has_dependents,-0.003344,0.005317
10,property_area,-0.003733,0.011903
4,education,-0.003774,0.006327


In [22]:
feature_importance["source_feature"] = (
    feature_importance["feature"]
    .str.replace(r"^[^_]+__", "", regex=True)
    .str.split("_")
    .str[0]
)

source_importance = (
    feature_importance
    .groupby("source_feature", as_index=False)["importance_mean"]
    .sum()
    .sort_values("importance_mean", ascending=False)
)

source_importance

,source_feature,importance_mean
2,credit,0.184144
1,coapplicant,0.013436
8,married,0.010708
10,self,0.002421
11,total,0.000103
0,applicant,-0.000944
3,debt,-0.002256
6,has,-0.003344
9,property,-0.003733
4,education,-0.003774


In [19]:
from sklearn.inspection import permutation_importance

permutation = permutation_importance(
    model_logistic,
    X_test_scaled,
    split["y_test"],
    scoring="roc_auc",
    n_repeats=30,
    random_state=RANDOM_SEED,
)

feature_names = encoders[
    "encoder_scaled"
].get_feature_names_out()

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

feature_importance.head(20)

,feature,importance_mean,importance_std
15,numeric__credit_history,0.184144,0.043670
12,numeric__coapplicant_income,0.013436,0.006953
2,onehot__married_No,0.005354,0.014562
3,onehot__married_Yes,0.005354,0.014562
8,onehot__property_area_Rural,0.001313,0.015368
7,onehot__self_employed_Yes,0.001210,0.003369
6,onehot__self_employed_No,0.001210,0.003369
17,numeric__total_income,0.000103,0.000229
11,numeric__applicant_income,-0.000944,0.001824
5,onehot__education_Not Graduate,-0.001682,0.006322


#### Ablation Testing

In [6]:
# Historical ablation testing: preserve the original baseline workflow
# These exploratory comparisons use the historical split and baseline specs.
# They remain for traceability and are not final holdout estimates.

from copy import deepcopy

from src.config import RANDOM_SEED

from src.data_config import FEATURE_COLUMNS, BASELINE_MODEL_SPECS
from src.model_pipeline import (
    build_encoder,
    train_logistic_regression,
    evaluate_classifier,
)

feature_sets = {
    "all_features": FEATURE_COLUMNS,

    "without_engineered_features": [
        column for column in FEATURE_COLUMNS
        if column not in {
            "total_income",
            "debt_income_ratio",
            "has_dependents",
        }
    ],

    "without_weak_demographics": [
        column for column in FEATURE_COLUMNS
        if column not in {
            "gender",
            "education",
            "property_area",
        }
    ],

    "without_loan_term": [
        column for column in FEATURE_COLUMNS
        if column != "loan_amount_term"
    ],

    "compact_core_features": [
        "credit_history",
        "applicant_income",
        "coapplicant_income",
        "loan_amount",
        "married",
        "self_employed",
    ],
}

In [7]:
def run_logistic_ablation(feature_columns):
    # Build each candidate from the baseline registry without mutating MODEL_SPECS.
    baseline_specs = deepcopy(BASELINE_MODEL_SPECS)
    original_spec = baseline_specs["classification"]
    candidate_specs = deepcopy(BASELINE_MODEL_SPECS)

    candidate_specs["classification"] = {
        **original_spec,
        "feature_columns": feature_columns,
        "encoding_map": {
            column: original_spec["encoding_map"][column]
            for column in feature_columns
        },
    }

    try:
        X_train = split["X_train"][feature_columns]
        X_test = split["X_test"][feature_columns]

        encoder = build_encoder(
            task="classification",
            scaled=True,
            specs=candidate_specs,
        )
        encoder.fit(X_train)

        X_train_encoded = encoder.transform(X_train)
        X_test_encoded = encoder.transform(X_test)

        model = train_logistic_regression(
            X_train_encoded,
            split["y_train"],
            random_state=RANDOM_SEED,
        )

        metrics = evaluate_classifier(
            model,
            X_test_encoded,
            split["y_test"],
            model_name="Logistic Regression",
        )

        return metrics

    finally:
        pass


In [8]:
ablation_results = []

for current_feature_set_name, current_feature_columns in feature_sets.items():
    metrics = run_logistic_ablation(current_feature_columns)

    ablation_results.append({
        "feature_set": current_feature_set_name,
        "feature_count": len(current_feature_columns),
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
    })

ablation_results_df = (
    pd.DataFrame(ablation_results)
    .sort_values(["roc_auc", "f1"], ascending=False)
    .reset_index(drop=True)
)

ablation_results_df

,feature_set,feature_count,accuracy,precision,recall,f1,roc_auc
0,compact_core_features,6,0.8,0.805195,0.953846,0.873239,0.744000
1,without_loan_term,12,0.8,0.805195,0.953846,0.873239,0.728615
2,without_engineered_features,10,0.8,0.805195,0.953846,0.873239,0.722462
3,without_weak_demographics,10,0.8,0.805195,0.953846,0.873239,0.720000
4,all_features,13,0.8,0.805195,0.953846,0.873239,0.715077


##### Cross-validation on Ablation test features

In [9]:
# Candidate feature sets are evaluated inside the historical training partition.
# The baseline registry keeps this analysis reproducible after final promotion.
from copy import deepcopy

import pandas as pd
from sklearn.model_selection import StratifiedKFold

from src.data_config import FEATURE_COLUMNS, BASELINE_MODEL_SPECS
from src.model_pipeline import (
    build_encoder,
    train_logistic_regression,
    evaluate_classifier,
)

feature_sets_for_cv = {
    "all_features": FEATURE_COLUMNS,

    "compact_core_features": [
        "credit_history",
        "applicant_income",
        "coapplicant_income",
        "loan_amount",
        "married",
        "self_employed",
    ],

    "without_loan_term": [
        column for column in FEATURE_COLUMNS
        if column != "loan_amount_term"
    ],
}

In [10]:
def cross_validate_feature_set(
    # Fit the encoder separately within each training fold to avoid leakage.
    feature_set_name,
    feature_columns,
    n_splits=5,
):
    baseline_specs = deepcopy(BASELINE_MODEL_SPECS)
    original_spec = baseline_specs["classification"]
    candidate_specs = deepcopy(BASELINE_MODEL_SPECS)

    candidate_specs["classification"] = {
        **original_spec,
        "feature_columns": feature_columns,
        "encoding_map": {
            column: original_spec["encoding_map"][column]
            for column in feature_columns
        },
    }

    X = split["X_train"][feature_columns]
    y = split["y_train"]

    stratified_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    fold_results = []

    try:
        for fold_number, (train_idx, validation_idx) in enumerate(
            stratified_cv.split(X, y),
            start=1,
        ):
            X_train_fold = X.iloc[train_idx]
            X_validation_fold = X.iloc[validation_idx]
            y_train_fold = y.iloc[train_idx]
            y_validation_fold = y.iloc[validation_idx]

            encoder = build_encoder(
                task="classification",
                scaled=True,
                specs=candidate_specs,
            )

            encoder.fit(X_train_fold)

            X_train_encoded = encoder.transform(X_train_fold)
            X_validation_encoded = encoder.transform(
                X_validation_fold
            )

            model = train_logistic_regression(
                X_train_encoded,
                y_train_fold,
                random_state=RANDOM_SEED,
            )

            metrics = evaluate_classifier(
                model,
                X_validation_encoded,
                y_validation_fold,
                model_name="Logistic Regression",
            )

            fold_results.append({
                "feature_set": feature_set_name,
                "fold": fold_number,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "roc_auc": metrics["roc_auc"],
            })

    finally:
        pass

    return fold_results

In [11]:
cv_fold_results = []

for current_feature_set_name, current_feature_columns in feature_sets_for_cv.items():
    cv_fold_results.extend(
        cross_validate_feature_set(
            current_feature_set_name,
            current_feature_columns,
            n_splits=5,
        )
    )

cv_fold_results_df = pd.DataFrame(cv_fold_results)
cv_fold_results_df

,feature_set,fold,accuracy,precision,recall,f1,roc_auc
0,all_features,1,0.791667,0.796875,0.962264,0.871795,0.671301
1,all_features,2,0.777778,0.790323,0.942308,0.859649,0.684615
2,all_features,3,0.791667,0.824561,0.903846,0.862385,0.741346
3,all_features,4,0.763889,0.777778,0.942308,0.852174,0.647115
4,all_features,5,0.791667,0.803279,0.942308,0.867257,0.724038
5,compact_core_features,1,0.791667,0.796875,0.962264,0.871795,0.659384
6,compact_core_features,2,0.791667,0.793651,0.961538,0.869565,0.661538
7,compact_core_features,3,0.791667,0.813559,0.923077,0.864865,0.745192
8,compact_core_features,4,0.763889,0.777778,0.942308,0.852174,0.623077
9,compact_core_features,5,0.791667,0.803279,0.942308,0.867257,0.767308


In [12]:
cv_summary = (
    cv_fold_results_df
    .groupby("feature_set")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
    )
    .sort_values(
        ["roc_auc_mean", "f1_mean"],
        ascending=False,
    )
    .reset_index()
)

cv_summary

,feature_set,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,all_features,0.783333,0.012423,0.798563,0.017327,0.938607,0.021267,0.862652,0.007478,0.693683,0.038552
1,without_loan_term,0.783333,0.012423,0.798563,0.017327,0.938607,0.021267,0.862652,0.007478,0.692939,0.033095
2,compact_core_features,0.786111,0.012423,0.797028,0.013175,0.946299,0.016265,0.865131,0.007690,0.691300,0.061726


## Post-Feature-Set Review and Final Holdout Setup:  

### Development Data 

This section is an additional, separate evaluation path. The earlier `split` workflow remains available for historical exploratory analysis. The new holdout is used only after training-only feature selection.

In [3]:
# Post-review development/holdout split: learned preprocessing values come from development data only.
from copy import deepcopy

from sklearn.model_selection import StratifiedKFold, train_test_split

from src.config import RANDOM_SEED

from src.data_config import (
    CATEGORICAL_FILL_VALUES,
    BASELINE_MODEL_SPECS,
    MODEL_SPECS,
    NUMERIC_FILL_VALUES,
    TARGET_COLUMN,
    FEATURE_COLUMNS,
)
from src.loans_preprocessing import run_cleaning_pipeline
from src.model_pipeline import (
    build_encoder,
    evaluate_classifier,
    train_logistic_regression,
)

POST_REVIEW_SPLIT_SEED = RANDOM_SEED + 1

if use_checkpoint("post_review_split_v2"):
    post_review_state = load_progress("post_review_split_v2")
    development_data = post_review_state["development_data"]
    final_holdout = post_review_state["final_holdout"]
    post_review_medians = post_review_state["medians"]
else:
    # Cleaning here performs validation and row removal only.
    # Learned numeric values are fitted after the split below.
    df_raw_review = load_data(Path(FILE_PATH))
    df_clean_review = run_cleaning_pipeline(df_raw_review)

    development_raw, final_holdout_raw = train_test_split(
        df_clean_review,
        test_size=0.20,
        random_state=POST_REVIEW_SPLIT_SEED,
        stratify=df_clean_review[TARGET_COLUMN],
    )

    post_review_medians = {
        column: development_raw[column].median()
        for column, fill_value in NUMERIC_FILL_VALUES.items()
        if fill_value == "median"
    }

    def apply_post_review_preprocessing(df, medians):
        result = df.copy()
        for column, fill_value in CATEGORICAL_FILL_VALUES.items():
            result[column] = result[column].fillna(fill_value)
        for column, fill_value in NUMERIC_FILL_VALUES.items():
            replacement = medians[column] if fill_value == "median" else fill_value
            result[column] = result[column].fillna(replacement)
        result["dependents"] = result["dependents"].astype(str).replace({"3+": "3"})
        result["has_dependents"] = (result["dependents"].astype(int) > 0).astype(int)
        result["total_income"] = result["applicant_income"] + result["coapplicant_income"]
        result["debt_income_ratio"] = result["total_income"] / result["loan_amount"]
        return result.drop(columns=["loan_id", "dependents"], errors="ignore")

    development_data = apply_post_review_preprocessing(
        development_raw, post_review_medians
    )
    final_holdout = apply_post_review_preprocessing(
        final_holdout_raw, post_review_medians
    )

    save_progress(
        "post_review_split_v2",
        {
            "development_data": development_data,
            "final_holdout": final_holdout,
            "medians": post_review_medians,
            "random_state": POST_REVIEW_SPLIT_SEED,
        },
    )

print(f"Post-review development data: {development_data.shape}")
print(f"Post-review final holdout:     {final_holdout.shape}")
print("Development target distribution:")
print(development_data[TARGET_COLUMN].value_counts(normalize=True).round(3))
print("Holdout target distribution:")
print(final_holdout[TARGET_COLUMN].value_counts(normalize=True).round(3))

Progress loaded: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_post_review_split_v2_20260910_191606.pkl
Post-review development data: (360, 14)
Post-review final holdout:     (90, 14)
Development target distribution:
loan_status
1.0    0.725
0.0    0.275
Name: proportion, dtype: float64
Holdout target distribution:
loan_status
1.0    0.722
0.0    0.278
Name: proportion, dtype: float64


### Training-Only Cross-Validation

In [14]:
post_review_feature_sets = {
    "all_features": FEATURE_COLUMNS,
    "without_loan_term": [
        column for column in FEATURE_COLUMNS
        if column != "loan_amount_term"
    ],
    "compact_core_features": [
        "credit_history",
        "applicant_income",
        "coapplicant_income",
        "loan_amount",
        "married",
        "self_employed",
    ],
}

def post_review_cross_validate(feature_set_name, feature_columns, n_splits=5):
    # Feature selection is based on development-only cross-validation.
    candidate_specs = deepcopy(BASELINE_MODEL_SPECS)
    original_spec = candidate_specs["classification"]
    candidate_specs["classification"] = {
        **original_spec,
        "feature_columns": feature_columns,
        "encoding_map": {
            column: original_spec["encoding_map"][column]
            for column in feature_columns
        },
    }

    X = development_data[feature_columns]
    y = development_data[TARGET_COLUMN]
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED,
    )
    results = []

    try:
        for fold, (train_idx, validation_idx) in enumerate(cv.split(X, y), start=1):
            X_train_fold = X.iloc[train_idx]
            X_validation_fold = X.iloc[validation_idx]
            y_train_fold = y.iloc[train_idx]
            y_validation_fold = y.iloc[validation_idx]

            encoder = build_encoder(
                task="classification",
                scaled=True,
                specs=candidate_specs,
            )
            encoder.fit(X_train_fold)
            model = train_logistic_regression(
                encoder.transform(X_train_fold),
                y_train_fold,
                random_state=RANDOM_SEED,
            )
            metrics = evaluate_classifier(
                model,
                encoder.transform(X_validation_fold),
                y_validation_fold,
                model_name="Post-review Logistic Regression",
            )
            results.append({
                "feature_set": feature_set_name,
                "fold": fold,
                "accuracy": metrics["accuracy"],
                "f1": metrics["f1"],
                "roc_auc": metrics["roc_auc"],
            })
    finally:
        pass

    return results

post_review_cv_results = []
for name, columns in post_review_feature_sets.items():
    post_review_cv_results.extend(
        post_review_cross_validate(name, columns)
    )

post_review_cv_results_df = pd.DataFrame(post_review_cv_results)
post_review_cv_summary = (
    post_review_cv_results_df
    .groupby("feature_set")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
    )
    .reset_index()
)
post_review_cv_summary

,feature_set,accuracy_mean,accuracy_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,all_features,0.786111,0.028801,0.864097,0.019058,0.615666,0.057137
1,compact_core_features,0.794444,0.026716,0.869686,0.017074,0.668579,0.076087
2,without_loan_term,0.788889,0.022822,0.866146,0.014800,0.630241,0.054875


### Final Holdout Evaluation

In [15]:
# Freeze the feature decision before using final_holdout.
# The final holdout is not consulted until this selection step is complete.
# Use a one-standard-error rule, then prefer the smaller feature set.
best_row = post_review_cv_summary.loc[
    post_review_cv_summary["roc_auc_mean"].idxmax()
]
roc_auc_threshold = (
    best_row["roc_auc_mean"]
    - best_row["roc_auc_std"] / (post_review_cv_results_df["fold"].nunique() ** 0.5)
)
eligible_feature_sets = post_review_cv_summary.loc[
    post_review_cv_summary["roc_auc_mean"] >= roc_auc_threshold
].copy()
eligible_feature_sets["feature_count"] = (
    eligible_feature_sets["feature_set"]
    .map(lambda name: len(post_review_feature_sets[name]))
)
selected_feature_set_name = (
    eligible_feature_sets
    .sort_values(["feature_count", "roc_auc_mean"], ascending=[True, False])
    .iloc[0]["feature_set"]
)
canonical_feature_set = post_review_feature_sets[selected_feature_set_name]

final_specs = deepcopy(BASELINE_MODEL_SPECS)
original_spec = final_specs["classification"]
final_specs["classification"] = {
    **original_spec,
    "feature_columns": canonical_feature_set,
    "encoding_map": {
        column: original_spec["encoding_map"][column]
        for column in canonical_feature_set
    },
}

final_encoder = build_encoder(
    task="classification",
    scaled=True,
    specs=final_specs,
)
final_encoder.fit(development_data[canonical_feature_set])
final_model = train_logistic_regression(
    final_encoder.transform(development_data[canonical_feature_set]),
    development_data[TARGET_COLUMN],
    random_state=RANDOM_SEED,
)
final_holdout_metrics = evaluate_classifier(
    final_model,
    final_encoder.transform(final_holdout[canonical_feature_set]),
    final_holdout[TARGET_COLUMN],
    model_name="Post-review final Logistic Regression",
)
post_review_final_state = {
    "selected_feature_set_name": selected_feature_set_name,
    "canonical_feature_set": canonical_feature_set,
    "cv_summary": post_review_cv_summary,
    "holdout_metrics": final_holdout_metrics,
    "random_state": POST_REVIEW_SPLIT_SEED,
}
save_progress("post_review_final_evaluation_v2", post_review_final_state)

print(f"Selected feature set: {selected_feature_set_name}")
print(f"Final holdout metrics: {final_holdout_metrics}")
final_holdout_metrics

Progress saved: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_post_review_final_evaluation_v2_20260911_103331.pkl
Selected feature set: compact_core_features
Final holdout metrics: {'model_name': 'Post-review final Logistic Regression', 'accuracy': 0.7888888888888889, 'precision': 0.7875, 'recall': 0.9692307692307692, 'f1': 0.8689655172413793, 'confusion_matrix': [[8, 17], [2, 63]], 'train_time_sec': None, 'roc_auc': 0.7310769230769232}


{'model_name': 'Post-review final Logistic Regression',
 'accuracy': 0.7888888888888889,
 'precision': 0.7875,
 'recall': 0.9692307692307692,
 'f1': 0.8689655172413793,
 'confusion_matrix': [[8, 17], [2, 63]],
 'train_time_sec': None,
 'roc_auc': 0.7310769230769232}

#### Interpretation note for interview and reporting
  
- Assuming `loan_status = 1` means approved, the compact model's holdout recall of 96.9% means it correctly identifies 63 of 65 approved cases and misses only 2.  
- This should not be interpreted as uniformly strong performance: it incorrectly predicts approval for 17 rejected cases, giving only 32.0% recall/specificity for the negative class.  
- Precision is 78.8%,  
- accuracy is 78.9%,  
- and ROC-AUC is 0.731.  
- The model is biased toward the positive class, so the business interpretation depends on whether missed approvals or incorrect approvals are more costly.  
- Threshold analysis, specificity, precision, and class-weighted alternatives should be considered before deployment.

### Source Completeness Context
  
This descriptive analysis uses the original raw dataset and the finalized compact feature set. It is reporting context only; it is not used for feature selection, model selection, or holdout evaluation.

In [16]:
# Measure raw-source completeness before cleaning or imputation for interview context.
import sys
from src.config import SRC_PATH

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from src.data_config import CLASSIFICATION_FEATURE_COLUMNS
from src.profiling_utils import feature_completeness_report

df_raw_completeness = load_data(Path(FILE_PATH))
source_completeness = feature_completeness_report(
    df_raw_completeness,
    CLASSIFICATION_FEATURE_COLUMNS,
)

save_progress(
    "source_completeness_context",
    source_completeness,
)

print(f"Raw rows assessed: {source_completeness['total_rows']}")
print(
    f"Complete across all six features: {source_completeness['complete_rows']} "
    f"({source_completeness['complete_pct']:.1f}%)"
)
print(
    f"Missing at least one feature: {source_completeness['incomplete_rows']} "
    f"({source_completeness['incomplete_pct']:.1f}%)"
)

source_completeness["per_feature"]

Progress saved: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_source_completeness_context_20260911_104446.pkl
Raw rows assessed: 563
Complete across all six features: 416 (73.9%)
Missing at least one feature: 147 (26.1%)


,feature,missing_count,missing_pct
0,credit_history,22,3.907638
1,applicant_income,26,4.618117
2,coapplicant_income,34,6.039076
3,loan_amount,30,5.328597
4,married,19,3.374778
5,self_employed,34,6.039076


## Canonical Final Evaluation: 

### Six-Feature Classification Results
  
This section uses the default `MODEL_SPECS` configuration. It is separate from the historical baseline workflow and does not require MLflow.

In [4]:
# Final canonical evaluation: use the promoted MODEL_SPECS on the untouched holdout.
# This section records final model comparisons after feature selection is complete.
from src.data_config import MODEL_SPECS, TARGET_COLUMN
from src.model_pipeline import (
    fit_and_checkpoint_encoders,
    train_decision_tree_classifier,
    train_gradient_boosting_classifier,
    train_logistic_regression,
    evaluate_classifier,
)

canonical_feature_columns = MODEL_SPECS["classification"]["feature_columns"]
assert len(canonical_feature_columns) == 6

canonical_encoders = fit_and_checkpoint_encoders(
    development_data[canonical_feature_columns],
    task="classification",
    specs=MODEL_SPECS,
)

canonical_X_development_scaled = canonical_encoders["encoder_scaled"].transform(
    development_data[canonical_feature_columns]
)
canonical_X_holdout_scaled = canonical_encoders["encoder_scaled"].transform(
    final_holdout[canonical_feature_columns]
)
canonical_X_development_raw = canonical_encoders["encoder_raw"].transform(
    development_data[canonical_feature_columns]
)
canonical_X_holdout_raw = canonical_encoders["encoder_raw"].transform(
    final_holdout[canonical_feature_columns]
)

canonical_models = {
    "Logistic Regression": (
        train_logistic_regression,
        canonical_X_development_scaled,
        canonical_X_holdout_scaled,
    ),
    "Decision Tree Classifier": (
        train_decision_tree_classifier,
        canonical_X_development_raw,
        canonical_X_holdout_raw,
    ),
    "Gradient Boosting Classifier": (
        train_gradient_boosting_classifier,
        canonical_X_development_raw,
        canonical_X_holdout_raw,
    ),
}

canonical_final_results = []
for model_name, (trainer, X_development, X_holdout) in canonical_models.items():
    model = trainer(
        X_development,
        development_data[TARGET_COLUMN],
        random_state=RANDOM_SEED,
    )
    metrics = evaluate_classifier(
        model,
        X_holdout,
        final_holdout[TARGET_COLUMN],
        model_name=f"Canonical {model_name}",
    )
    canonical_final_results.append(metrics)

canonical_final_results_df = (
    pd.DataFrame(canonical_final_results)
    .sort_values(["roc_auc", "f1"], ascending=False)
    .reset_index(drop=True)
)

canonical_final_state = {
    "feature_columns": canonical_feature_columns,
    "results": canonical_final_results_df,
    "random_state": RANDOM_SEED,
}
save_progress("canonical_final_pipeline", canonical_final_state)
canonical_final_results_df

Scaled encoder saved: Q:\scripts\projects\Mock_Interview-July2026\outputs\models\classification_encoder_fitted_scaled_20260911_111356.pkl
Raw encoder saved: Q:\scripts\projects\Mock_Interview-July2026\outputs\models\classification_encoder_fitted_raw_20260911_111356.pkl
Progress saved: Q:\scripts\projects\Mock_Interview-July2026\exports\loan_canonical_final_pipeline_20260911_111356.pkl


,model_name,accuracy,precision,recall,f1,confusion_matrix,train_time_sec,roc_auc
0,Canonical Logistic Regression,0.788889,0.787500,0.969231,0.868966,"[[8, 17], [2, 63]]",None,0.731077
1,Canonical Gradient Boosting Classifier,0.800000,0.797468,0.969231,0.875000,"[[9, 16], [2, 63]]",None,0.675077
2,Canonical Decision Tree Classifier,0.700000,0.779412,0.815385,0.796992,"[[10, 15], [12, 53]]",None,0.607692


## Historical Baseline and Canonical Evaluation Comparison

The project retains the original 13-feature baseline results as a historical reference.  
Those models were trained and evaluated using the earlier split and preprocessing workflow.  
  
The canonical evaluation uses the finalized six-feature classification set:  
  
- `credit_history`
- `applicant_income`
- `coapplicant_income`
- `loan_amount`
- `married`
- `self_employed`
  
The canonical models were evaluated once on the corrected clean holdout:  
  
| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |  
|---|---:|---:|---:|---:|---:|  
| Historical Logistic Regression | 0.800 | 0.805 | 0.954 | 0.873 | 0.715 |  
| Canonical Logistic Regression | 0.789 | 0.788 | 0.969 | 0.869 | 0.731 |  
| Canonical Gradient Boosting | 0.800 | 0.797 | 0.969 | 0.875 | 0.675 |  
| Canonical Decision Tree | 0.700 | 0.779 | 0.815 | 0.797 | 0.608 |  
  
The historical and canonical results are useful for contextual comparison, but they should not be interpreted as a controlled experiment because they use different data splits and evaluation workflows.  
The canonical results are the appropriate final estimates for this project because the feature set was selected using training-only cross-validation and the final holdout was reserved for final evaluation.  
  
The canonical Logistic Regression model provides the clearest final reference:  
- it identifies 96.9% of approved cases,  
- with 78.8% precision,  
- 78.9% accuracy,  
- and a ROC-AUC of 0.731 (slightly better than GB).  
  
### Canonical Model Confusion Matrices

Rows represent actual outcomes; columns represent predicted outcomes. Approved cases are shown first.

#### Canonical Logistic Regression

|              |    |   Predicted   |   Predicted   |     |
| ------------ | -- | :-----------: | :-----------: | :-: |
| Actual       |    | Approved (1)  | Rejected (0)  |     |
| Approved (1) | TP |      63       |       2       | FN  |
| Rejected (0) | FP |      17       |       8       | TN  |

#### Canonical Gradient Boosting Classifier

|              |    |   Predicted   |   Predicted   |     |
| ------------ | -- | :-----------: | :-----------: | :-: |
| Actual       |    | Approved (1)  | Rejected (0)  |     |
| Approved (1) | TP |      63       |       2       | FN  |
| Rejected (0) | FP |      16       |       9       | TN  |

#### Canonical Decision Tree Classifier

|              |    |   Predicted   |   Predicted   |     |
| ------------ | -- | :-----------: | :-----------: | :-: |
| Actual       |    | Approved (1)  | Rejected (0)  |     |
| Approved (1) | TP |      53       |      12       | FN  |
| Rejected (0) | FP |      15       |      10       | TN  |  
  
This high approval recall should be interpreted together with rejected-case performance:   
- the model produces false approvals,  
- so recall alone does not represent overall decision quality.  